# Homework Starter — Stage 05: Data Storage
Name: Huaiwen Dong
Date: 2026.08.17

Objectives:
- Env-driven paths to `data/raw/` and `data/processed/`
- Save CSV and Parquet; reload and validate
- Abstract IO with utility functions; document choices

In [1]:
# --- packages this notebook needs (uncomment and run once, then re-comment) ---
# !pip install numpy
# !pip install pandas
# !pip install pyarrow
# !pip install python-dotenv

In [2]:
# --- files this notebook needs (run me first - I only report, I change nothing) ---
from pathlib import Path

ROOT = Path.cwd().parent        # notebooks are meant to be run from their own folder
CHECKS = [
    (".env", "NEEDED", "YOU create this: copy .env.example to .env. Missing = no error, but the config demo silently shows nothing"),
    (".env.example", "NEEDED", "shipped with this stage - the template you copy to .env"),
]

print(f"Looking in: {ROOT}\n")
missing = 0
for rel, kind, note in CHECKS:
    here = (ROOT / rel).exists()
    if not here and kind == "NEEDED":
        missing += 1
    print(f"  [{'OK ' if here else 'MISS'}]  {kind:<8}  {rel:<34}  {note}")

if missing:
    print(f"\n{missing} needed file(s) missing. Put them at the paths above, relative to:\n  {ROOT}")
    print("If that folder looks wrong, you are running the notebook from the wrong place.")
else:
    print("\nAll needed files present.")

Looking in: /Users/juan/Desktop/nyu bootcamp/bootcamp_henry_dong

  [OK ]  NEEDED    .env                                YOU create this: copy .env.example to .env. Missing = no error, but the config demo silently shows nothing
  [OK ]  NEEDED    .env.example                        shipped with this stage - the template you copy to .env

All needed files present.


In [3]:
import os
import pathlib
import datetime as dt

import pandas as pd
from dotenv import load_dotenv

# Load environment variables from the project root
load_dotenv(ROOT / ".env")

RAW = ROOT / os.getenv("DATA_DIR_RAW", "data/raw")
PROC = ROOT / os.getenv("DATA_DIR_PROCESSED", "data/processed")

# Create directories if they do not exist
RAW.mkdir(parents=True, exist_ok=True)
PROC.mkdir(parents=True, exist_ok=True)

print("RAW ->", RAW.resolve())
print("PROC ->", PROC.resolve())

RAW -> /Users/juan/Desktop/nyu bootcamp/bootcamp_henry_dong/data/raw
PROC -> /Users/juan/Desktop/nyu bootcamp/bootcamp_henry_dong/data/processed


## 1) Create or Load a Sample DataFrame
You may reuse data from prior stages or create a small synthetic dataset.

In [4]:
import numpy as np

rng = np.random.default_rng(42)

dates = pd.date_range(
    "2024-01-01",
    periods=20,
    freq="D"
)

df = pd.DataFrame({
    "date": dates,
    "ticker": ["AAPL"] * 20,
    "price": 150 + rng.normal(size=20).cumsum()
})

df.head()

,date,ticker,price
0,2024-01-01,AAPL,150.304717
1,2024-01-02,AAPL,149.264733
2,2024-01-03,AAPL,150.015184
3,2024-01-04,AAPL,150.955749
4,2024-01-05,AAPL,149.004714


## 2) Save CSV to data/raw/ and Parquet to data/processed/ (TODO)
- Use timestamped filenames.
- Handle missing Parquet engine gracefully.

In [5]:
def ts():
    return dt.datetime.now().strftime("%Y%m%d-%H%M")

stamp = ts()

csv_path = RAW / f"sample_{stamp}.csv"
pq_path = PROC / f"sample_{stamp}.parquet"


# Save CSV to data/raw/
df.to_csv(
    csv_path,
    index=False
)

print("CSV saved:", csv_path)


# Save Parquet to data/processed/
try:
    df.to_parquet(
        pq_path,
        index=False
    )

    print("Parquet saved:", pq_path)

except ImportError as e:
    print(
        "Parquet engine not available. "
        "Install pyarrow or fastparquet."
    )
    pq_path = None

CSV saved: /Users/juan/Desktop/nyu bootcamp/bootcamp_henry_dong/data/raw/sample_20260819-1213.csv
Parquet saved: /Users/juan/Desktop/nyu bootcamp/bootcamp_henry_dong/data/processed/sample_20260819-1213.parquet


## 3) Reload and Validate (TODO)
- Compare shapes and key dtypes.

In [6]:
# Reload CSV and Parquet
df_csv = pd.read_csv(
    csv_path,
    parse_dates=["date"]
)

df_parquet = pd.read_parquet(
    pq_path
)

print("CSV shape:", df_csv.shape)
print("Parquet shape:", df_parquet.shape)

print("\nCSV dtypes:")
print(df_csv.dtypes)

print("\nParquet dtypes:")
print(df_parquet.dtypes)

CSV shape: (20, 3)
Parquet shape: (20, 3)

CSV dtypes:
date      datetime64[us]
ticker               str
price            float64
dtype: object

Parquet dtypes:
date      datetime64[us]
ticker               str
price            float64
dtype: object


In [7]:
def validate_reloaded_data(df_csv, df_parquet):
    results = {}

    # Shape check
    results["shape_match"] = df_csv.shape == df_parquet.shape

    # Required columns
    required_cols = {"date", "ticker", "price"}
    results["required_columns"] = (
        required_cols.issubset(df_csv.columns)
        and required_cols.issubset(df_parquet.columns)
    )

    # Date dtype
    results["date_is_datetime"] = (
        pd.api.types.is_datetime64_any_dtype(df_csv["date"])
        and pd.api.types.is_datetime64_any_dtype(df_parquet["date"])
    )

    # Price dtype
    results["price_is_numeric"] = (
        pd.api.types.is_numeric_dtype(df_csv["price"])
        and pd.api.types.is_numeric_dtype(df_parquet["price"])
    )

    # Ticker dtype
    results["ticker_is_text"] = (
        pd.api.types.is_string_dtype(df_csv["ticker"])
        and pd.api.types.is_string_dtype(df_parquet["ticker"])
    )

    return results


validation_results = validate_reloaded_data(
    df_csv,
    df_parquet
)

for check, passed in validation_results.items():
    print(f"{check}: {passed}")

assert all(validation_results.values()), \
    "One or more validation checks failed."

print("\nReload validation passed.")

shape_match: True
required_columns: True
date_is_datetime: True
price_is_numeric: True
ticker_is_text: True

Reload validation passed.


## 4) Utilities (TODO)
- Implement `detect_format`, `write_df`, `read_df`.
- Use suffix to route; create parent dirs if needed; friendly errors for Parquet.

In [8]:
from pathlib import Path


def detect_format(path):
    """
    Detect file format from the file suffix.
    """
    suffix = Path(path).suffix.lower()

    if suffix == ".csv":
        return "csv"

    elif suffix == ".parquet":
        return "parquet"

    else:
        raise ValueError(
            f"Unsupported file format: {suffix}. "
            "Use .csv or .parquet."
        )


def write_df(df, path):
    """
    Write a DataFrame to CSV or Parquet based on file suffix.
    Creates the parent directory if it does not exist.
    """
    path = Path(path)

    # Create missing directories automatically
    path.parent.mkdir(parents=True, exist_ok=True)

    file_format = detect_format(path)

    if file_format == "csv":
        df.to_csv(path, index=False)

    elif file_format == "parquet":
        try:
            df.to_parquet(path, index=False)
        except ImportError as e:
            raise ImportError(
                "Parquet engine is not available. "
                "Install pyarrow or fastparquet."
            ) from e

    print(f"Written: {path}")
    return path


def read_df(path, **kwargs):
    """
    Read a CSV or Parquet file based on file suffix.
    """
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(
            f"File does not exist: {path}"
        )

    file_format = detect_format(path)

    if file_format == "csv":
        return pd.read_csv(path, **kwargs)

    elif file_format == "parquet":
        try:
            return pd.read_parquet(path, **kwargs)
        except ImportError as e:
            raise ImportError(
                "Parquet engine is not available. "
                "Install pyarrow or fastparquet."
            ) from e

In [9]:
# Test write_df using the existing paths
write_df(df, csv_path)
write_df(df, pq_path)

# Test read_df
df_csv_util = read_df(
    csv_path,
    parse_dates=["date"]
)

df_parquet_util = read_df(
    pq_path
)

print("\nCSV loaded with utility:")
print(df_csv_util.head())

print("\nParquet loaded with utility:")
print(df_parquet_util.head())

Written: /Users/juan/Desktop/nyu bootcamp/bootcamp_henry_dong/data/raw/sample_20260819-1213.csv
Written: /Users/juan/Desktop/nyu bootcamp/bootcamp_henry_dong/data/processed/sample_20260819-1213.parquet

CSV loaded with utility:
        date ticker       price
0 2024-01-01   AAPL  150.304717
1 2024-01-02   AAPL  149.264733
2 2024-01-03   AAPL  150.015184
3 2024-01-04   AAPL  150.955749
4 2024-01-05   AAPL  149.004714

Parquet loaded with utility:
        date ticker       price
0 2024-01-01   AAPL  150.304717
1 2024-01-02   AAPL  149.264733
2 2024-01-03   AAPL  150.015184
3 2024-01-04   AAPL  150.955749
4 2024-01-05   AAPL  149.004714


In [10]:
assert df_csv_util.shape == df.shape, \
    "CSV utility reload shape does not match."

assert df_parquet_util.shape == df.shape, \
    "Parquet utility reload shape does not match."

assert list(df_csv_util.columns) == list(df.columns), \
    "CSV columns do not match."

assert list(df_parquet_util.columns) == list(df.columns), \
    "Parquet columns do not match."

print("Utility read/write validation passed.")

Utility read/write validation passed.


## 5) Documentation (TODO)
- Update README with a **Data Storage** section (folders, formats, env usage).
- Summarize validation checks and any assumptions.